# Visualising elfe3D_GPR Results and Post Processing

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pyvista as pv
from scipy.interpolate import griddata

In [2]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
postprocess_folder = os.path.join(base_folder, "postprocess")
analytical_folder = os.path.join(base_folder, "semi-analytic_100MHz")

z_value = 0.01
z_tol = 0.05
xlim=(-2.25, 5.75)
ylim=(-2.25, 5.75)
xlim_base = (-5, 5)
ylim_base = (-5, 5)
xlim_a = (-1.5, 5)
ylim_a = (-1.5, 5)
z_tol=0.05
dpi=300
grid_resolution=3000
amplitude_cmap='Blues_r'
phase_cmap='RdBu_r'

component = 'Ex' 

## Super Fine Mesh

### Air

In [129]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_air_half_PML_surf_feng.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_l20.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_l20_min.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3]

In [130]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
# Mask grid_x and grid_y to be within xlim_a and ylim_a
mask_x = (grid_x >= xlim_a[0]) & (grid_x <= xlim_a[1])
mask_y = (grid_y >= ylim_a[0]) & (grid_y <= ylim_a[1])
mask = mask_x & mask_y

# Mask grid_x and grid_y to be within xlim_a and ylim_a, and flatten to get valid points
valid_mask = mask.flatten()
grid_x_bounded = grid_x.flatten()[valid_mask]
grid_y_bounded = grid_y.flatten()[valid_mask]

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')

# Resize analytical grids to match grid_x and grid_y shape, filling extra entries with NaN
amp_analytical_grid_full = np.full(grid_x.shape, np.nan)
phase_analytical_grid_full = np.full(grid_x.shape, np.nan)
real_analytical_grid_full = np.full(grid_x.shape, np.nan)
imag_analytical_grid_full = np.full(grid_x.shape, np.nan)

amp_analytical_grid_full.flat[valid_mask] = amp_analytical_grid
phase_analytical_grid_full.flat[valid_mask] = phase_analytical_grid
real_analytical_grid_full.flat[valid_mask] = real_analytical_grid
imag_analytical_grid_full.flat[valid_mask] = imag_analytical_grid

amp_analytical_grid = amp_analytical_grid_full
phase_analytical_grid = phase_analytical_grid_full
real_analytical_grid = real_analytical_grid_full
imag_analytical_grid = imag_analytical_grid_full

# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'Default PML, $\lambda/10$ Elements',
                 r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/8$ PML and z, $\lambda/20$ Elements'
                 ]
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
axs[0, n_entries].add_patch(rect1)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
axs[1, n_entries].add_patch(rect2)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Fine Mesh Based Truncation, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"1_fine_mesh_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [131]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'Default PML, $\lambda/10$ Elements',
                 r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/8$ PML and z, $\lambda/20$ Elements'
                 ]
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Amplitude-Clipped Fine Mesh Based Truncation, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"2_fine_mesh_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [132]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'Default PML, $\lambda/10$ Elements',
                 r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/8$ PML and z, $\lambda/20$ Elements'
                 ]
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'% Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = (diff_amp / amp_analytical_grid) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Mesh Truncation and Element Size\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "3_all_mesh_size_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


In [133]:
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
diff_amp_percent_all = []
diff_phase_all = []
std_amp_all = []
std_phase_all = []
amp_text_list = [r'Default PML, $\lambda/10$ Elements',
                 r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/8$ PML and z, $\lambda/20$ Elements'
                 ]
phase_text_list = amp_text_list.copy()
for i in range(n_entries):
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_amp_percent = np.clip(diff_amp_percent, -100, 100)
    diff_phase_percent = (phase_elfe_grid_all[i] - phase_analytical_grid) * 100 
    diff_phase_percent = np.clip(diff_phase_percent, -100, 100)
    std_amp_all.append(np.nanstd(diff_amp_percent))
    std_phase_all.append(np.nanstd(diff_phase_percent))

    diff_amp_percent_all.append(diff_amp_percent)
    diff_phase_all.append(diff_phase_percent)

# Plotting the histograms
fig, axs = plt.subplots(2, n_entries, figsize=(6.5 * n_entries, 14), dpi=dpi)
for i in range(n_entries):
    axs[0, i].hist(diff_amp_percent_all[i].flatten(), bins=100, color='blue', alpha=0.7)
    axs[0, i].set_title(f'Amp Diff %: {amp_text_list[i]}', fontsize=18, fontweight='bold')
    axs[0, i].set_xlabel('Difference (%)', fontsize=16)
    axs[0, i].set_ylabel('Frequency', fontsize=16)
    axs[0, i].tick_params(axis='both', which='major', labelsize=16)
    axs[0, i].axvline(std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].axvline(-std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].text(std_amp_all[i], axs[0, i].get_ylim()[1]*0.9, f'STD: {std_amp_all[i]:.2f}', color='blue', fontsize=16)
    axs[0, i].set_xlabel("x", fontsize=16)
    axs[0, i].set_ylabel("y", fontsize=16)
    axs[0, i].tick_params(axis='both', which='major', labelsize=14)

    axs[1, i].hist(diff_phase_all[i].flatten(), bins=100, color='red', alpha=0.7)
    axs[1, i].set_title(f'Phase Diff %: {phase_text_list[i]}', fontsize=18, fontweight='bold')
    axs[1, i].set_xlabel('Difference (%)', fontsize=16)
    axs[1, i].set_ylabel('Frequency', fontsize=16)
    axs[1, i].tick_params(axis='both', which='major', labelsize=16)
    axs[1, i].axvline(std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].axvline(-std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].text(std_phase_all[i], axs[1, i].get_ylim()[1]*0.9, f'STD: {std_phase_all[i]:.2f}', color='red', fontsize=16)
    axs[1, i].set_xlabel("x", fontsize=16)
    axs[1, i].set_ylabel("y", fontsize=16)
    axs[1, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.89])
plt.suptitle('Histogram of Differences Between elfe3D and Analytical Solution for Various Mesh Truncation and Element Size\nFrom a 2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=24, fontweight='bold')
plt.savefig(os.path.join(postprocess_folder, "4_mesh_truncation_sizes_histogram.png"), dpi=dpi)
plt.close(fig)


### Half-Space

In [134]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
electric_file = os.path.join(base_folder, "out_4_3m_air", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)


e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, || DBC',
    r'$\lambda/4$ PML, 3m Air, || DBC'
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, x_an, y_an, abs_an, phase_an, real_an, imag_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0, 0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[0, 1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1, 0].plot(rec_distance_an, real_an, linestyle='-', color='black', label='Analytical Real Part', linewidth=2)
    axes[1, 1].plot(rec_distance_an[2:], imag_an[2:], linestyle='-', color='black', label='Analytical Imaginary Part', linewidth=2)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_mesh_truncation_sizes_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_broadside[1:]
phase_Ex_an_ef = phase_broadside[1:]
real_Ex_an_ef = real_broadside[1:]
imag_Ex_an_ef = imag_broadside[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_endfire[1:]
phase_Ex_an_bs = phase_endfire[1:]
real_Ex_an_bs = real_endfire[1:]
imag_Ex_an_bs = imag_endfire[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs, 'Broadside', 5, '1_4_analytic', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 6, '1_4_analytic', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob, 'Oblique, 45°', 7, '1_4_analytic', labels)

In [138]:
def plot_diff_ana_receiver_line(component,
                                    rec_x_list, rec_y_list,
                                    abs_data_list, phase_data_list,
                                    real_data_list, imag_data_list,
                                    x_an, y_an,
                                    abs_an, phase_an, real_an, imag_an,
                                    type_rec, index, purpose, labels,
                                    postprocess_folder):
    """
    Plot percent‐difference between each ELFE run and the analytical data
    for a given receiver‐line type (Broadside / Endfire / Oblique).
    """

    # 1) Set up 2×2 figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 2) Decide which coordinate is “distance” for this receiver type
    if type_rec == 'Broadside':
        coord_label = 'y'
        rec_distance_an = np.array(y_an)
    elif type_rec == 'Endfire':
        coord_label = 'x'
        rec_distance_an = np.array(x_an)
    else:  # 'Oblique, 45°'
        coord_label = 'radial distance along 45° from origin'
        rec_distance_an = np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)

    # 3) Style arrays (you can tweak these if you like)
    colors     = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers    = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8
    line_width  = 1.5

    # Ensure analytical arrays are sorted by rec_distance_an
    sort_idx = np.argsort(rec_distance_an)
    rd_an_sorted    = rec_distance_an[sort_idx]
    abs_an_sorted   = abs_an[sort_idx]
    phase_an_sorted = phase_an[sort_idx]
    real_an_sorted  = real_an[sort_idx]
    imag_an_sorted  = imag_an[sort_idx]

    # 4) Loop over each ELFE dataset
    for i, (abs_elfe, phase_elfe, real_elfe, imag_elfe,
            rec_xi, rec_yi, label) in enumerate(
            zip(abs_data_list, phase_data_list,
                real_data_list, imag_data_list,
                rec_x_list, rec_y_list, labels)):

        # 4.1) Compute ELFE “distance” array for this receiver line
        if type_rec == 'Broadside':
            rec_distance = rec_yi
        elif type_rec == 'Endfire':
            rec_distance = rec_xi
        else:
            rec_distance = np.sqrt(rec_xi**2 + rec_yi**2)

        # 4.2) Interpolate analytical onto ELFE distances
        # (np.interp requires ascending x, so we use rd_an_sorted)
        abs_an_interp   = np.interp(rec_distance, rd_an_sorted, abs_an_sorted)
        phase_an_interp = np.interp(rec_distance, rd_an_sorted, phase_an_sorted)
        real_an_interp  = np.interp(rec_distance, rd_an_sorted, real_an_sorted)
        imag_an_interp  = np.interp(rec_distance, rd_an_sorted, imag_an_sorted)

        # 4.3) Compute percent differences (avoid divide-by-zero by replacing denom=0 → np.nan)
        pct_abs   = (abs_elfe   - abs_an_interp)
        pct_phase = (phase_elfe - phase_an_interp)
        pct_real  = 100.0 * (real_elfe  - real_an_interp)  / np.where(real_an_interp  == 0, np.nan, real_an_interp)
        pct_imag  = 100.0 * (imag_elfe  - imag_an_interp)  / np.where(imag_an_interp  == 0, np.nan, imag_an_interp)

        # 4.4) Plot each Δ%
        style_idx = i % len(colors)
        axes[0, 0].plot(rec_distance, pct_abs,
                   linestyle=linestyles[style_idx],
                   marker=markers[style_idx],
                   markersize=marker_size,
                   color=colors[style_idx],
                   label=label,
                   linewidth=line_width)

        axes[0, 1].plot(rec_distance, pct_phase * (180/np.pi),
                   linestyle=linestyles[style_idx],
                   marker=markers[style_idx],
                   markersize=marker_size,
                   color=colors[style_idx],
                   label=label,
                   linewidth=line_width)

        axes[1, 0].plot(rec_distance, pct_real,
                   linestyle=linestyles[style_idx],
                   marker=markers[style_idx],
                   markersize=marker_size,
                   color=colors[style_idx],
                   label=label,
                   linewidth=line_width)

        axes[1, 1].plot(rec_distance, pct_imag,
                   linestyle=linestyles[style_idx],
                   marker=markers[style_idx],
                   markersize=marker_size,
                   color=colors[style_idx],
                   label=label,
                   linewidth=line_width)


    # 5) Label axes, set titles, grid, legend
    axes[0, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    # Use |Ex| notation without \bigl/\bigr
    axes[0, 0].set_ylabel(r'$\Delta\;|'+component+r'|$', fontsize=12)
    axes[0, 0].set_title(f'Ratio of Log of Amplitude', fontsize=16)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='best')

    axes[0, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 1].set_ylabel(r'$\Delta\;\angle('+component+r')$', fontsize=12)
    axes[0, 1].set_title(f'Difference of Phase({component})', fontsize=16)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 0].set_ylabel(r'$\Delta\%\;\mathrm{Re}('+component+r')$', fontsize=12)
    axes[1, 0].set_title(f'%‐Diff of Re({component})', fontsize=16)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 1].set_ylabel(r'$\Delta\%\;\mathrm{Im}('+component+r')$', fontsize=12)
    axes[1, 1].set_title(f'%‐Diff of Im({component})', fontsize=16)
    axes[1, 1].grid(True)

    plt.suptitle(
        f'Percent Difference vs Analytical (EmpyMod)\n'
        f'{type_rec} Receiver Line → Component: {component}',
        fontsize=18, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 6) Save to disk
    fname = f'{index}_{purpose}_percent_diff_{type_rec.replace(" ", "_")}.png'
    out_path = os.path.join(postprocess_folder, fname)
    plt.savefig(out_path, dpi=300)
    plt.close(fig)


# Broadside percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_bs_list, rec_y_bs_list,
    abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
    x_an_bs, y_an_bs,
    abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    type_rec='Broadside',
    index=5,      # matches your naming convention
    purpose='2_4_diff_analytic',  # matches your “purpose” string
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Endfire percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_ef_list, rec_y_ef_list,
    abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
    x_an_ef, y_an_ef,
    abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    type_rec='Endfire',
    index=6,
    purpose='2_4_diff_analytic',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Oblique (45°) percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_ob_list, rec_y_ob_list,
    abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
    x_an_ob, y_an_ob,
    abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    type_rec='Oblique, 45°',
    index=7,
    purpose='2_4_diff_analytic',
    labels=labels,
    postprocess_folder=postprocess_folder
)


Half-space with empymod

In [119]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
electric_file = os.path.join(base_folder, "out_4_3m_air", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, || DBC',
    r'$\lambda/4$ PML, 3m Air, || DBC',
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "halfspace_model_results_empymod.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = -1*analytical_lines[:, 7]
imag_broadside = -1*analytical_lines[:, 8]
complex_broadside = real_broadside + 1j * imag_broadside
abs_broadside = np.abs(complex_broadside)
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = -1*analytical_lines[:, 11]
imag_45deg = -1*analytical_lines[:, 12]
complex_45deg = real_45deg + 1j * imag_45deg
abs_45deg = np.abs(complex_45deg)
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = -1*analytical_lines[:, 3]
imag_endfire = -1*analytical_lines[:, 4]
complex_endfire = real_endfire + 1j * imag_endfire
abs_endfire = np.abs(complex_endfire)
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, x_an, y_an, abs_an, phase_an, real_an, imag_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0, 0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[0, 1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1, 0].plot(rec_distance_an, real_an, linestyle='-', color='black', label='Analytical Real Part', linewidth=2)
    axes[1, 1].plot(rec_distance_an[2:], imag_an[2:], linestyle='-', color='black', label='Analytical Imaginary Part', linewidth=2)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_mesh_truncation_sizes_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_endfire[1:]
phase_Ex_an_ef = phase_endfire[1:]
real_Ex_an_ef = real_endfire[1:]
imag_Ex_an_ef = imag_endfire[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_broadside[1:]
phase_Ex_an_bs = phase_broadside[1:]
real_Ex_an_bs = real_broadside[1:]
imag_Ex_an_bs = imag_broadside[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs, 'Broadside', 5, '3_4_emp', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 6, '3_4_emp', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob, 'Oblique, 45°', 7, '3_4_emp', labels)

In [120]:
def plot_diff_emp_receiver_line(component,
                                    rec_x_list, rec_y_list,
                                    abs_data_list, phase_data_list,
                                    real_data_list, imag_data_list,
                                    x_an, y_an,
                                    abs_an, phase_an, real_an, imag_an,
                                    type_rec, index, purpose, labels,
                                    postprocess_folder):
    """
    Plot percent‐difference between each ELFE line and the analytical (Empymod) line.
    
    Parameters
    ----------
    component : str
        Name of the field component (e.g. 'Ex').
    rec_x_list, rec_y_list : list of 1D‐arrays
        Lists (length = n_ELFE_runs) of x‐ and y‐coordinates of ELFE receivers.
    abs_data_list, phase_data_list, real_data_list, imag_data_list : list of 1D‐arrays
        Lists (length = n_ELFE_runs) of absolute, phase, real, imag from ELFE.
    x_an, y_an : 1D‐arrays
        Analytical receiver coordinates (same length as abs_an, etc.).
    abs_an, phase_an, real_an, imag_an : 1D‐arrays
        Analytical solution (absolute, phase [rad], real, imag) on the grid x_an/y_an.
    type_rec : str
        One of {'Broadside', 'Endfire', 'Oblique, 45°'}. Used to label axes.
    index : int
        Integer index used to construct the output filename.
    purpose : str
        String used in output filename (e.g. '4_emp').
    labels : list of str
        Labels (length = n_ELFE_runs) for each ELFE curve.
    postprocess_folder : str
        Directory where the PNG will be saved.
    """
    # Create 2×2 grid of subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Decide which coordinate to treat as “distance”
    if type_rec == 'Broadside':
        coord_label = 'y'
        # analytical rec‐distance is simply y_an
        rec_distance_an = np.array(y_an)
    elif type_rec == 'Endfire':
        coord_label = 'x'
        rec_distance_an = np.array(x_an)
    else:
        coord_label = 'radial distance along 45° from origin'
        rec_distance_an = np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    
    # styling arrays (reuse your original colors/linestyles/markers)
    colors    = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers   = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8
    line_width  = 1.5
    
    # Loop over each ELFE run
    for i, (abs_elfe, phase_elfe, real_elfe, imag_elfe,
            rec_xi, rec_yi, label) in enumerate(
            zip(abs_data_list, phase_data_list,
                real_data_list, imag_data_list,
                rec_x_list, rec_y_list, labels)):
        
        # 1) Build ELFE “receiver distance” array:
        if type_rec == 'Broadside':
            rec_distance = rec_yi
        elif type_rec == 'Endfire':
            rec_distance = rec_xi
        else:
            rec_distance = np.sqrt(rec_xi**2 + rec_yi**2)
        
        # 2) Interpolate analytical quantities onto ELFE rec_distance:
        #    We assume rec_distance_an is sorted. If not, sort + apply same sorting to abs_an, etc.
        #    np.interp requires ascending x values → ensure rec_distance_an is ascending:
        sort_idx = np.argsort(rec_distance_an)
        rd_an_sorted = rec_distance_an[sort_idx]
        abs_an_sorted   = abs_an[sort_idx]
        phase_an_sorted = phase_an[sort_idx]
        real_an_sorted  = real_an[sort_idx]
        imag_an_sorted  = imag_an[sort_idx]
        
        # Now interpolate (if rec_distance goes outside [min, max] of rd_an_sorted → extrapolation):
        abs_an_interp   = np.interp(rec_distance, rd_an_sorted, abs_an_sorted)
        phase_an_interp = np.interp(rec_distance, rd_an_sorted, phase_an_sorted)
        real_an_interp  = np.interp(rec_distance, rd_an_sorted, real_an_sorted)
        imag_an_interp  = np.interp(rec_distance, rd_an_sorted, imag_an_sorted)
        
        # 3) Compute percent differences:
        #    Avoid division by zero: wherever abs_an_interp == 0, set percent difference to NaN.
        pct_abs = (abs_elfe - abs_an_interp)
        pct_phase = (phase_elfe - phase_an_interp)
        pct_real  = 100.0 * (real_elfe - real_an_interp) / np.where(real_an_interp == 0, np.nan, real_an_interp)
        pct_imag  = 100.0 * (imag_elfe - imag_an_interp) / np.where(imag_an_interp == 0, np.nan, imag_an_interp)
        
        # 4) Plot Δ%@Abs on axes[0,0], Δ%@Phase on axes[0,1], etc.
        style_idx = i % len(colors)
        
        axes[0, 0].plot(rec_distance, pct_abs,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[0, 1].plot(rec_distance, pct_phase * (180/np.pi),  # convert phase diff (rad)→deg before percent?
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[1, 0].plot(rec_distance, pct_real,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[1, 1].plot(rec_distance, pct_imag,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
    
    # 5) Axis labels, titles, grid, legend
    axes[0, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 0].set_ylabel(r'Ratio of $\log_{10},|'+component+r'|$ Components', fontsize=12)
    axes[0, 0].set_title(f'Difference of |{component}|', fontsize=16)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='best')
    
    axes[0, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 1].set_ylabel(r'$\Delta,\angle('+component+r')$', fontsize=12)
    axes[0, 1].set_title(f'Difference of Phase({component})', fontsize=16)
    axes[0, 1].grid(True)
    
    axes[1, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 0].set_ylabel(r'$\Delta\%\,\mathrm{Re}('+component+r')$', fontsize=12)
    axes[1, 0].set_title(f'Percent Difference of Re({component})', fontsize=16)
    axes[1, 0].grid(True)
    
    axes[1, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 1].set_ylabel(r'$\Delta\%\,\mathrm{Im}('+component+r')$', fontsize=12)
    axes[1, 1].set_title(f'Percent Difference of Im({component})', fontsize=16)
    axes[1, 1].grid(True)
    
    plt.suptitle(
        f'Percent Difference vs Analytical (EmpyMod)\n'
        f'{type_rec} Receiver Line, Component = {component}',
        fontsize=18, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # 6) Save figure
    plot_name = f'{index}_{purpose}_percent_diff_{type_rec.replace(" ", "_")}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close(fig)


# Broadside percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_bs_list, rec_y_bs_list,
    abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
    x_an_bs, y_an_bs,
    abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    type_rec='Broadside',
    index=5,
    purpose='4_4_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Endfire percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_ef_list, rec_y_ef_list,
    abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
    x_an_ef, y_an_ef,
    abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    type_rec='Endfire',
    index=6,
    purpose='4_4_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Oblique (45°) percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_ob_list, rec_y_ob_list,
    abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
    x_an_ob, y_an_ob,
    abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    type_rec='Oblique, 45°',
    index=7,
    purpose='4_4_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)


### Two Layers

Evert

In [121]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_9_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
electric_file = os.path.join(base_folder, "out_4_9_3m_air", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)


e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, || DBC',
    r'$\lambda/4$ PML, 3m Air, || DBC'
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)


# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_broadside[1:]
phase_Ex_an_ef = phase_broadside[1:]
real_Ex_an_ef = real_broadside[1:]
imag_Ex_an_ef = imag_broadside[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_endfire[1:]
phase_Ex_an_bs = phase_endfire[1:]
real_Ex_an_bs = real_endfire[1:]
imag_Ex_an_bs = imag_endfire[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs, 'Broadside', 8, '1_4_9_analytic', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 9, '1_4_9_analytic', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob, 'Oblique, 45°', 10, '1_4_9_analytic', labels)

In [122]:
def plot_diff_ana_receiver_line(component,
                                    rec_x_list, rec_y_list,
                                    abs_data_list, phase_data_list,
                                    real_data_list, imag_data_list,
                                    x_an, y_an,
                                    abs_an, phase_an, real_an, imag_an,
                                    type_rec, index, purpose, labels,
                                    postprocess_folder):
    """
    Plot percent‐difference between each ELFE run and the analytical data
    for a given receiver‐line type (Broadside / Endfire / Oblique).
    """

    # 1) Set up 2×2 figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 2) Decide which coordinate is “distance” for this receiver type
    if type_rec == 'Broadside':
        coord_label = 'y'
        rec_distance_an = np.array(y_an)
    elif type_rec == 'Endfire':
        coord_label = 'x'
        rec_distance_an = np.array(x_an)
    else:  # 'Oblique, 45°'
        coord_label = 'radial distance along 45° from origin'
        rec_distance_an = np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)

    # 3) Style arrays (you can tweak these if you like)
    colors     = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers    = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8
    line_width  = 1.5

    # Ensure analytical arrays are sorted by rec_distance_an
    sort_idx = np.argsort(rec_distance_an)
    rd_an_sorted    = rec_distance_an[sort_idx]
    abs_an_sorted   = abs_an[sort_idx]
    phase_an_sorted = phase_an[sort_idx]
    real_an_sorted  = real_an[sort_idx]
    imag_an_sorted  = imag_an[sort_idx]

    # 4) Loop over each ELFE dataset
    for i, (abs_elfe, phase_elfe, real_elfe, imag_elfe,
            rec_xi, rec_yi, label) in enumerate(
            zip(abs_data_list, phase_data_list,
                real_data_list, imag_data_list,
                rec_x_list, rec_y_list, labels)):

        # 4.1) Compute ELFE “distance” array for this receiver line
        if type_rec == 'Broadside':
            rec_distance = rec_yi
        elif type_rec == 'Endfire':
            rec_distance = rec_xi
        else:
            rec_distance = np.sqrt(rec_xi**2 + rec_yi**2)

        # 4.2) Interpolate analytical onto ELFE distances
        # (np.interp requires ascending x, so we use rd_an_sorted)
        abs_an_interp   = np.interp(rec_distance, rd_an_sorted, abs_an_sorted)
        phase_an_interp = np.interp(rec_distance, rd_an_sorted, phase_an_sorted)
        real_an_interp  = np.interp(rec_distance, rd_an_sorted, real_an_sorted)
        imag_an_interp  = np.interp(rec_distance, rd_an_sorted, imag_an_sorted)

        # 4.3) Compute percent differences (avoid divide-by-zero by replacing denom=0 → np.nan)
        pct_abs   = (abs_elfe   - abs_an_interp)
        pct_phase = (phase_elfe - phase_an_interp)
        pct_real  = 100.0 * (real_elfe  - real_an_interp)  / np.where(real_an_interp  == 0, np.nan, real_an_interp)
        pct_imag  = 100.0 * (imag_elfe  - imag_an_interp)  / np.where(imag_an_interp  == 0, np.nan, imag_an_interp)

        # 4.4) Plot each Δ%
        style_idx = i % len(colors)
        axes[0, 0].plot(rec_distance, pct_abs,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)

        # Convert phase‐diff from rad→deg before showing percent (optional)
        axes[0, 1].plot(rec_distance, pct_phase * (180/np.pi),
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)

        axes[1, 0].plot(rec_distance, pct_real,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)

        axes[1, 1].plot(rec_distance, pct_imag,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)

    # 5) Label axes, set titles, grid, legend
    axes[0, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    # Use |Ex| notation without \bigl/\bigr
    axes[0, 0].set_ylabel(r'$\Delta\;|'+component+r'|$', fontsize=12)
    axes[0, 0].set_title(f'Ratio of Log of Amplitude', fontsize=16)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='best')

    axes[0, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 1].set_ylabel(r'$\Delta\;\angle('+component+r')$', fontsize=12)
    axes[0, 1].set_title(f'Difference of Phase({component})', fontsize=16)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 0].set_ylabel(r'$\Delta\%\;\mathrm{Re}('+component+r')$', fontsize=12)
    axes[1, 0].set_title(f'%‐Diff of Re({component})', fontsize=16)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 1].set_ylabel(r'$\Delta\%\;\mathrm{Im}('+component+r')$', fontsize=12)
    axes[1, 1].set_title(f'%‐Diff of Im({component})', fontsize=16)
    axes[1, 1].grid(True)

    plt.suptitle(
        f'Percent Difference vs Analytical (EmpyMod)\n'
        f'{type_rec} Receiver Line → Component: {component}',
        fontsize=18, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 6) Save to disk
    fname = f'{index}_{purpose}_percent_diff_{type_rec.replace(" ", "_")}.png'
    out_path = os.path.join(postprocess_folder, fname)
    plt.savefig(out_path, dpi=300)
    plt.close(fig)



# Broadside percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_bs_list, rec_y_bs_list,
    abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
    x_an_bs, y_an_bs,
    abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    type_rec='Broadside',
    index=8,      # matches your naming convention
    purpose='2_4_9_diff_analytic',  # matches your “purpose” string
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Endfire percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_ef_list, rec_y_ef_list,
    abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
    x_an_ef, y_an_ef,
    abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    type_rec='Endfire',
    index=9,
    purpose='2_4_9_diff_analytic',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Oblique (45°) percent‐difference
plot_diff_ana_receiver_line(
    'Ex',
    rec_x_ob_list, rec_y_ob_list,
    abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
    x_an_ob, y_an_ob,
    abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    type_rec='Oblique, 45°',
    index=10,
    purpose='2_4_9_diff_analytic',
    labels=labels,
    postprocess_folder=postprocess_folder
)


In [123]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_9_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"

electric_file = os.path.join(base_folder, "out_4_9_3m_air", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, || DBC',
    r'$\lambda/4$ PML, 3m Air, || DBC',
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "two_layered_model_results_empymod.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = -1*analytical_lines[:, 7]
imag_broadside = -1*analytical_lines[:, 8]
complex_broadside = real_broadside + 1j * imag_broadside
abs_broadside = np.abs(complex_broadside)
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = -1*analytical_lines[:, 11]
imag_45deg = -1*analytical_lines[:, 12]
complex_45deg = real_45deg + 1j * imag_45deg
abs_45deg = np.abs(complex_45deg)
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = -1*analytical_lines[:, 3]
imag_endfire = -1*analytical_lines[:, 4]
complex_endfire = real_endfire + 1j * imag_endfire
abs_endfire = np.abs(complex_endfire)
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, x_an, y_an, abs_an, phase_an, real_an, imag_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0, 0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[0, 1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1, 0].plot(rec_distance_an, real_an, linestyle='-', color='black', label='Analytical Real Part', linewidth=2)
    axes[1, 1].plot(rec_distance_an[2:], imag_an[2:], linestyle='-', color='black', label='Analytical Imaginary Part', linewidth=2)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_mesh_truncation_sizes_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_endfire[1:]
phase_Ex_an_ef = phase_endfire[1:]
real_Ex_an_ef = real_endfire[1:]
imag_Ex_an_ef = imag_endfire[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_broadside[1:]
phase_Ex_an_bs = phase_broadside[1:]
real_Ex_an_bs = real_broadside[1:]
imag_Ex_an_bs = imag_broadside[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs, 'Broadside', 8, '3_4_9_emp', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 9, '3_4_9_emp', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob, 'Oblique, 45°', 10, '3_4_9_emp', labels)

In [124]:
def plot_diff_emp_receiver_line(component,
                                    rec_x_list, rec_y_list,
                                    abs_data_list, phase_data_list,
                                    real_data_list, imag_data_list,
                                    x_an, y_an,
                                    abs_an, phase_an, real_an, imag_an,
                                    type_rec, index, purpose, labels,
                                    postprocess_folder):
    """
    Plot percent‐difference between each ELFE line and the analytical (Empymod) line.
    
    Parameters
    ----------
    component : str
        Name of the field component (e.g. 'Ex').
    rec_x_list, rec_y_list : list of 1D‐arrays
        Lists (length = n_ELFE_runs) of x‐ and y‐coordinates of ELFE receivers.
    abs_data_list, phase_data_list, real_data_list, imag_data_list : list of 1D‐arrays
        Lists (length = n_ELFE_runs) of absolute, phase, real, imag from ELFE.
    x_an, y_an : 1D‐arrays
        Analytical receiver coordinates (same length as abs_an, etc.).
    abs_an, phase_an, real_an, imag_an : 1D‐arrays
        Analytical solution (absolute, phase [rad], real, imag) on the grid x_an/y_an.
    type_rec : str
        One of {'Broadside', 'Endfire', 'Oblique, 45°'}. Used to label axes.
    index : int
        Integer index used to construct the output filename.
    purpose : str
        String used in output filename (e.g. '4_emp').
    labels : list of str
        Labels (length = n_ELFE_runs) for each ELFE curve.
    postprocess_folder : str
        Directory where the PNG will be saved.
    """
    # Create 2×2 grid of subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Decide which coordinate to treat as “distance”
    if type_rec == 'Broadside':
        coord_label = 'y'
        # analytical rec‐distance is simply y_an
        rec_distance_an = np.array(y_an)
    elif type_rec == 'Endfire':
        coord_label = 'x'
        rec_distance_an = np.array(x_an)
    else:
        coord_label = 'radial distance along 45° from origin'
        rec_distance_an = np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    
    # styling arrays (reuse your original colors/linestyles/markers)
    colors    = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers   = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8
    line_width  = 1.5
    
    # Loop over each ELFE run
    for i, (abs_elfe, phase_elfe, real_elfe, imag_elfe,
            rec_xi, rec_yi, label) in enumerate(
            zip(abs_data_list, phase_data_list,
                real_data_list, imag_data_list,
                rec_x_list, rec_y_list, labels)):
        
        # 1) Build ELFE “receiver distance” array:
        if type_rec == 'Broadside':
            rec_distance = rec_yi
        elif type_rec == 'Endfire':
            rec_distance = rec_xi
        else:
            rec_distance = np.sqrt(rec_xi**2 + rec_yi**2)
        
        # 2) Interpolate analytical quantities onto ELFE rec_distance:
        #    We assume rec_distance_an is sorted. If not, sort + apply same sorting to abs_an, etc.
        #    np.interp requires ascending x values → ensure rec_distance_an is ascending:
        sort_idx = np.argsort(rec_distance_an)
        rd_an_sorted = rec_distance_an[sort_idx]
        abs_an_sorted   = abs_an[sort_idx]
        phase_an_sorted = phase_an[sort_idx]
        real_an_sorted  = real_an[sort_idx]
        imag_an_sorted  = imag_an[sort_idx]
        
        # Now interpolate (if rec_distance goes outside [min, max] of rd_an_sorted → extrapolation):
        abs_an_interp   = np.interp(rec_distance, rd_an_sorted, abs_an_sorted)
        phase_an_interp = np.interp(rec_distance, rd_an_sorted, phase_an_sorted)
        real_an_interp  = np.interp(rec_distance, rd_an_sorted, real_an_sorted)
        imag_an_interp  = np.interp(rec_distance, rd_an_sorted, imag_an_sorted)
        
        # 3) Compute percent differences:
        #    Avoid division by zero: wherever abs_an_interp == 0, set percent difference to NaN.
        pct_abs = (abs_elfe - abs_an_interp)
        pct_phase = (phase_elfe - phase_an_interp)
        pct_real  = 100.0 * (real_elfe - real_an_interp) / np.where(real_an_interp == 0, np.nan, real_an_interp)
        pct_imag  = 100.0 * (imag_elfe - imag_an_interp) / np.where(imag_an_interp == 0, np.nan, imag_an_interp)
        
        # 4) Plot Δ%@Abs on axes[0,0], Δ%@Phase on axes[0,1], etc.
        style_idx = i % len(colors)
        
        axes[0, 0].plot(rec_distance, pct_abs,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[0, 1].plot(rec_distance, pct_phase * (180/np.pi),  # convert phase diff (rad)→deg before percent?
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[1, 0].plot(rec_distance, pct_real,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
        
        axes[1, 1].plot(rec_distance, pct_imag,
                       linestyle=linestyles[style_idx],
                       marker=markers[style_idx],
                       markersize=marker_size,
                       color=colors[style_idx],
                       label=label,
                       linewidth=line_width)
    
    # 5) Axis labels, titles, grid, legend
    axes[0, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 0].set_ylabel(r'Ratio of $\log_{10},|'+component+r'|$ Components', fontsize=12)
    axes[0, 0].set_title(f'Difference of |{component}|', fontsize=16)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='best')
    
    axes[0, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[0, 1].set_ylabel(r'$\Delta,\angle('+component+r')$', fontsize=12)
    axes[0, 1].set_title(f'Difference of Phase({component})', fontsize=16)
    axes[0, 1].grid(True)
    
    axes[1, 0].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 0].set_ylabel(r'$\Delta\%\,\mathrm{Re}('+component+r')$', fontsize=12)
    axes[1, 0].set_title(f'Percent Difference of Re({component})', fontsize=16)
    axes[1, 0].grid(True)
    
    axes[1, 1].set_xlabel(f'{coord_label} (m)', fontsize=12)
    axes[1, 1].set_ylabel(r'$\Delta\%\,\mathrm{Im}('+component+r')$', fontsize=12)
    axes[1, 1].set_title(f'Percent Difference of Im({component})', fontsize=16)
    axes[1, 1].grid(True)
    
    plt.suptitle(
        f'Percent Difference vs Analytical (EmpyMod)\n'
        f'{type_rec} Receiver Line, Component = {component}',
        fontsize=18, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # 6) Save figure
    plot_name = f'{index}_{purpose}_percent_diff_{type_rec.replace(" ", "_")}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close(fig)




# Broadside percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_bs_list, rec_y_bs_list,
    abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list,
    x_an_bs, y_an_bs,
    abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    type_rec='Broadside',
    index=8,
    purpose='4_4_9_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Endfire percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_ef_list, rec_y_ef_list,
    abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
    x_an_ef, y_an_ef,
    abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    type_rec='Endfire',
    index=9,
    purpose='4_4_9_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)

# Oblique (45°) percent‐difference plot
plot_diff_emp_receiver_line(
    'Ex',
    rec_x_ob_list, rec_y_ob_list,
    abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list,
    x_an_ob, y_an_ob,
    abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    type_rec='Oblique, 45°',
    index=10,
    purpose='4_4_9_diff_emp',
    labels=labels,
    postprocess_folder=postprocess_folder
)


## EMPyMod vs Evert's Solution

In [125]:
# EMPyMod
empymod_file = os.path.join(analytical_folder, "halfspace_model_results_empymod.csv")
empymod_lines = np.loadtxt(empymod_file, delimiter=',', skiprows=1)

# Extract columns by header order for EMPyMod
r_empymod = empymod_lines[:, 0]
amp_ef = empymod_lines[:, 1]
phase_ef = empymod_lines[:, 2]
re_ef = -1*empymod_lines[:, 3]
im_ef = -1*empymod_lines[:, 4]
cm_ef = re_ef + 1j * im_ef
phase_ef = np.angle(cm_ef)  # returns phase in radians (−π to π)

amp_bs = empymod_lines[:, 5]
phase_bs = empymod_lines[:, 6]
re_bs = -1*empymod_lines[:, 7]
im_bs = -1*empymod_lines[:, 8]
cm_bs = re_bs + 1j * im_bs
phase_bs = np.angle(cm_bs)  # returns phase in radians (−π to π)

amp_ob = empymod_lines[:, 9]
phase_ob = empymod_lines[:, 10]
re_ob = -1*empymod_lines[:, 11]
im_ob = -1*empymod_lines[:, 12]
cm_ob = re_ob + 1j * im_ob
phase_ob = np.angle(cm_ob)  # returns phase in radians (−π to π)

# Evert
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[2:]
y_an_ef = np.zeros_like(r[2:])
abs_Ex_an_ef = abs_broadside[2:]
phase_Ex_an_ef = phase_broadside[2:]
real_Ex_an_ef = real_broadside[2:]
imag_Ex_an_ef = imag_broadside[2:]

x_an_bs = np.zeros_like(r[2:])
y_an_bs = r[2:]
abs_Ex_an_bs = abs_endfire[2:]
phase_Ex_an_bs = phase_endfire[2:]
real_Ex_an_bs = real_endfire[2:]
imag_Ex_an_bs = imag_endfire[2:]

x_an_ob = r[2:]
y_an_ob = r[2:]
abs_Ex_an_ob = abs_45deg[2:]
phase_Ex_an_ob = phase_45deg[2:]
real_Ex_an_ob = real_45deg[2:]
imag_Ex_an_ob = imag_45deg[2:]

# Plot EMPyMod results vs Analytical Solution

def plot_empymod_vs_analytical(x, abs_emp, phase_emp, real_emp, imag_emp,
                               x_an, abs_an, phase_an, real_an, imag_an, label, fname):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs[0, 0].plot(x, abs_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[0, 0].plot(x_an, abs_an, 'k-', label='Analytical', linewidth=2)
    axs[0, 0].set_yscale('log')
    axs[0, 0].set_xlabel('Distance (m)')
    axs[0, 0].set_ylabel('Abs')
    axs[0, 0].set_title(f'Absolute Value ({label})')
    axs[0, 0].legend()
    axs[0, 0].grid(True)

    axs[0, 1].plot(x, phase_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[0, 1].plot(x_an, phase_an, 'k-', label='Analytical', linewidth=2)
    axs[0, 1].set_xlabel('Distance (m)')
    axs[0, 1].set_ylabel('Phase (rad)')
    axs[0, 1].set_title(f'Phase ({label})')
    axs[0, 1].legend()
    axs[0, 1].grid(True)

    axs[1, 0].plot(x, real_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[1, 0].plot(x_an, real_an, 'k-', label='Analytical', linewidth=2)
    axs[1, 0].set_xlabel('Distance (m)')
    axs[1, 0].set_ylabel('Real')
    axs[1, 0].set_title(f'Real Part ({label})')
    axs[1, 0].legend()
    axs[1, 0].grid(True)

    axs[1, 1].plot(x, imag_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[1, 1].plot(x_an, imag_an, 'k-', label='Analytical', linewidth=2)
    axs[1, 1].set_xlabel('Distance (m)')
    axs[1, 1].set_ylabel('Imag')
    axs[1, 1].set_title(f'Imaginary Part ({label})')
    axs[1, 1].legend()
    axs[1, 1].grid(True)
    plt.tight_layout()
    plt.suptitle(f'EMPYMOD vs Analytical - {label}', fontsize=18, fontweight='bold', y=1.04)
    plt.savefig(os.path.join(postprocess_folder, fname), dpi=300)
    plt.close(fig)

# Broadside
plot_empymod_vs_analytical(
    r_empymod, amp_bs, phase_bs, re_bs, im_bs,
    y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    'Broadside', '11_4_empymod_vs_analytical_broadside.png'
)

# Endfire
plot_empymod_vs_analytical(
    r_empymod, amp_ef, phase_ef, re_ef, im_ef,
    x_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    'Endfire', '12_4_empymod_vs_analytical_endfire.png'
)

# Oblique
plot_empymod_vs_analytical(
    r_empymod, amp_ob, phase_ob, re_ob, im_ob,
    x_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    'Oblique', '13_4_empymod_vs_analytical_oblique.png'
)

In [126]:
def line_differences(r_other_line, r_analytical_line,
                             amp_other_line, phase_other_line, real_other_line, imag_other_line, 
                             amp_analytical_line, phase_analytical_line, real_analytical_line, imag_analytical_line):
    
    # Find the minimum bounds between the two lines
    min_r = max(min(r_other_line), min(r_analytical_line))
    max_r = min(max(r_other_line), max(r_analytical_line))

    # Find the line indices that are within the bounds
    idx_other = np.where((r_other_line >= min_r) & (r_other_line <= max_r))[0]
    idx_analytical = np.where((r_analytical_line >= min_r) & (r_analytical_line <= max_r))[0]
    print(f"Bounds: {min_r} to {max_r}")

    # Select the data within the bounds
    r_other_line = r_other_line[idx_other]
    r_analytical_line = r_analytical_line[idx_analytical]
    amp_other_line = amp_other_line[idx_other]
    amp_analytical_line = amp_analytical_line[idx_analytical]
    phase_other_line = phase_other_line[idx_other]
    phase_analytical_line = phase_analytical_line[idx_analytical]
    real_other_line = real_other_line[idx_other]
    real_analytical_line = real_analytical_line[idx_analytical]
    imag_other_line = imag_other_line[idx_other]
    imag_analytical_line = imag_analytical_line[idx_analytical]

    # Create a new array of r values that has 100 points between the bounds
    r = np.linspace(min_r, max_r, 100)

    # Interpolate both lines to the new r values
    amp_analytical_line = np.interp(r, r_analytical_line, amp_analytical_line)
    phase_analytical_line = np.interp(r, r_analytical_line, phase_analytical_line)
    real_analytical_line = np.interp(r, r_analytical_line, real_analytical_line)
    imag_analytical_line = np.interp(r, r_analytical_line, imag_analytical_line)

    amp_other_line = np.interp(r, r_other_line, amp_other_line)
    phase_other_line = np.interp(r, r_other_line, phase_other_line)
    real_other_line = np.interp(r, r_other_line, real_other_line)
    imag_other_line = np.interp(r, r_other_line, imag_other_line)

    diff_amp_percent = ((amp_other_line - amp_analytical_line) / amp_analytical_line) * 100
    diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100
    diff_phase_percent = (phase_other_line - phase_analytical_line) * 100
    diff_real_percent = ((real_other_line - real_analytical_line) / real_analytical_line) * 100
    diff_imag_percent = ((imag_other_line - imag_analytical_line) / imag_analytical_line) * 100

    analytic_data = np.column_stack((amp_analytical_line, phase_analytical_line, real_analytical_line, imag_analytical_line))
    other_data = np.column_stack((amp_other_line, phase_other_line, real_other_line, imag_other_line))
    diff_data = np.column_stack((diff_amp_percent, diff_amp_percent_power, diff_phase_percent, diff_real_percent, diff_imag_percent))

    return r, analytic_data, other_data, diff_data



def plot_differences_line(diff_amp, diff_phase, diff_real, diff_imag, r, type_rec, index, purpose):
    """
    Plot differences in amplitude, phase, real, and imaginary parts.
    """
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Amplitude difference
    axes[0, 0].plot(r, diff_amp, linestyle='-', color='blue', label='Amplitude Difference (%)', linewidth=2)
    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel('Amplitude Difference (%)', fontsize=12)
    axes[0, 0].set_title('Amplitude Difference', fontsize=18)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    # Phase difference
    axes[0, 1].plot(r, diff_phase, linestyle='-', color='orange', label='Phase Difference (%)', linewidth=2)
    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel('Phase Difference (%)', fontsize=12)
    axes[0, 1].set_title('Phase Difference', fontsize=18)
    axes[0, 1].grid(True)

    # Real part difference
    if diff_real is not None:
        axes[1, 0].plot(r, diff_real, linestyle='-', color='green', label='Real Part Difference (%)', linewidth=2)
        axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
        axes[1, 0].set_ylabel('Real Part Difference (%)', fontsize=12)
        axes[1, 0].set_title('Real Part Difference', fontsize=18)
        axes[1, 0].grid(True)

    # Imaginary part difference
    if diff_imag is not None:
        axes[1, 1].plot(r, diff_imag, linestyle='-', color='red', label='Imaginary Part Difference (%)', linewidth=2)
        axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
        axes[1, 1].set_ylabel('Imaginary Part Difference (%)', fontsize=12)
        axes[1, 1].set_title('Imaginary Part Difference', fontsize=18)
    if diff_real is not None:
        axes[1, 1].grid(True)
    else:
        axes[1, 1].axis('off')
    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data Differences'.format(type_rec)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{type_rec}_differences_empymod.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Compute differences for each receiver type
r_diff_ef, analytic_ef, other_ef, diff_ef = line_differences(
    r_empymod, r[2:], amp_ef, phase_ef, re_ef, im_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef)
Evert_ef_amp, Evert_ef_phase, Evert_ef_re, Evert_ef_im = analytic_ef[:, 0], analytic_ef[:, 1], analytic_ef[:, 2], analytic_ef[:, 3]
EM_ef_amp, EM_ef_phase, EM_ef_re, EM_ef_im = other_ef[:, 0], other_ef[:, 1], other_ef[:, 2], other_ef[:, 3]
Diff_ef_amp, Diff_ef_amp10, Diff_ef_phase, Diff_ef_re, Diff_ef_im = diff_ef[:, 0], diff_ef[:, 1], diff_ef[:, 2], diff_ef[:, 3], diff_ef[:, 4]

r_diff_bs, analytic_bs, other_bs, diff_bs = line_differences(
    r_empymod, r[2:], amp_bs, phase_bs, re_bs, im_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs)
Evert_bs_amp, Evert_bs_phase, Evert_bs_re, Evert_bs_im = analytic_bs[:, 0], analytic_bs[:, 1], analytic_bs[:, 2], analytic_bs[:, 3]
EM_bs_amp, EM_bs_phase, EM_bs_re, EM_bs_im = other_bs[:, 0], other_bs[:, 1], other_bs[:, 2], other_bs[:, 3]
Diff_bs_amp, Diff_bs_amp10, Diff_bs_phase, Diff_bs_re, Diff_bs_im = diff_bs[:, 0], diff_bs[:, 1], diff_bs[:, 2], diff_bs[:, 3], diff_bs[:, 4]

r_diff_ob, analytic_ob, other_ob, diff_ob = line_differences(
    r_empymod, r[2:], amp_ob, phase_ob, re_ob, im_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob)
Evert_ob_amp, Evert_ob_phase, Evert_ob_re, Evert_ob_im = analytic_ob[:, 0], analytic_ob[:, 1], analytic_ob[:, 2], analytic_ob[:, 3]
EM_ob_amp, EM_ob_phase, EM_ob_re, EM_ob_im = other_ob[:, 0], other_ob[:, 1], other_ob[:, 2], other_ob[:, 3]
Diff_ob_amp, Diff_ob_amp10, Diff_ob_phase, Diff_ob_re, Diff_ob_im = diff_ob[:, 0], diff_ob[:, 1], diff_ob[:, 2], diff_ob[:, 3], diff_ob[:, 4]

# Plot differences for each receiver type
plot_differences_line(
    Diff_ef_amp, Diff_ef_phase, Diff_ef_re, Diff_ef_im, r_diff_ef, 'Endfire', 14, '4_empymod_vs_analytical')
plot_differences_line(
    Diff_bs_amp, Diff_bs_phase, Diff_bs_re, Diff_bs_im, r_diff_bs, 'Broadside', 15, '4_empymod_vs_analytical')
plot_differences_line(
    Diff_ob_amp, Diff_ob_phase, Diff_ob_re, Diff_ob_im, r_diff_ob, 'Oblique', 16, '4_empymod_vs_analytical')

Bounds: 0.2 to 4.75
Bounds: 0.2 to 4.75
Bounds: 0.2 to 4.75


C:\Users\Knight\AppData\Local\Temp\ipykernel_13772\1270505074.py:41: RuntimeWarning: overflow encountered in power
  diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100
C:\Users\Knight\AppData\Local\Temp\ipykernel_13772\1270505074.py:41: RuntimeWarning: invalid value encountered in subtract
  diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100


Two Layered Comparison

In [127]:
# EMPyMod
empymod_file = os.path.join(analytical_folder, "two_layered_model_results_empymod.csv")
empymod_lines = np.loadtxt(empymod_file, delimiter=',', skiprows=1)

# Extract columns by header order for EMPyMod
r_empymod = empymod_lines[:, 0]
amp_ef = empymod_lines[:, 1]
phase_ef = empymod_lines[:, 2]
re_ef = -1*empymod_lines[:, 3]
im_ef = -1*empymod_lines[:, 4]
cm_ef = re_ef + 1j * im_ef
phase_ef = np.angle(cm_ef)  # returns phase in radians (−π to π)

amp_bs = empymod_lines[:, 5]
phase_bs = empymod_lines[:, 6]
re_bs = -1*empymod_lines[:, 7]
im_bs = -1*empymod_lines[:, 8]
cm_bs = re_bs + 1j * im_bs
phase_bs = np.angle(cm_bs)  # returns phase in radians (−π to π)

amp_ob = empymod_lines[:, 9]
phase_ob = empymod_lines[:, 10]
re_ob = -1*empymod_lines[:, 11]
im_ob = -1*empymod_lines[:, 12]
cm_ob = re_ob + 1j * im_ob
phase_ob = np.angle(cm_ob)  # returns phase in radians (−π to π)

# Evert
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[2:]
y_an_ef = np.zeros_like(r[2:])
abs_Ex_an_ef = abs_broadside[2:]
phase_Ex_an_ef = phase_broadside[2:]
real_Ex_an_ef = real_broadside[2:]
imag_Ex_an_ef = imag_broadside[2:]

x_an_bs = np.zeros_like(r[2:])
y_an_bs = r[2:]
abs_Ex_an_bs = abs_endfire[2:]
phase_Ex_an_bs = phase_endfire[2:]
real_Ex_an_bs = real_endfire[2:]
imag_Ex_an_bs = imag_endfire[2:]

x_an_ob = r[2:]
y_an_ob = r[2:]
abs_Ex_an_ob = abs_45deg[2:]
phase_Ex_an_ob = phase_45deg[2:]
real_Ex_an_ob = real_45deg[2:]
imag_Ex_an_ob = imag_45deg[2:]

# Plot EMPyMod results vs Analytical Solution

def plot_empymod_vs_analytical(x, abs_emp, phase_emp, real_emp, imag_emp,
                               x_an, abs_an, phase_an, real_an, imag_an, label, fname):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs[0, 0].plot(x, abs_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[0, 0].plot(x_an, abs_an, 'k-', label='Analytical', linewidth=2)
    axs[0, 0].set_yscale('log')
    axs[0, 0].set_xlabel('Distance (m)')
    axs[0, 0].set_ylabel('Abs')
    axs[0, 0].set_title(f'Absolute Value ({label})')
    axs[0, 0].legend()
    axs[0, 0].grid(True)

    axs[0, 1].plot(x, phase_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[0, 1].plot(x_an, phase_an, 'k-', label='Analytical', linewidth=2)
    axs[0, 1].set_xlabel('Distance (m)')
    axs[0, 1].set_ylabel('Phase (rad)')
    axs[0, 1].set_title(f'Phase ({label})')
    axs[0, 1].legend()
    axs[0, 1].grid(True)

    axs[1, 0].plot(x, real_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[1, 0].plot(x_an, real_an, 'k-', label='Analytical', linewidth=2)
    axs[1, 0].set_xlabel('Distance (m)')
    axs[1, 0].set_ylabel('Real')
    axs[1, 0].set_title(f'Real Part ({label})')
    axs[1, 0].legend()
    axs[1, 0].grid(True)

    axs[1, 1].plot(x, imag_emp, '-.', color='purple', label='empymod', linewidth=3)
    axs[1, 1].plot(x_an, imag_an, 'k-', label='Analytical', linewidth=2)
    axs[1, 1].set_xlabel('Distance (m)')
    axs[1, 1].set_ylabel('Imag')
    axs[1, 1].set_title(f'Imaginary Part ({label})')
    axs[1, 1].legend()
    axs[1, 1].grid(True)
    plt.tight_layout()
    plt.suptitle(f'EMPYMOD vs Analytical - {label}', fontsize=18, fontweight='bold', y=1.04)
    plt.savefig(os.path.join(postprocess_folder, fname), dpi=300)
    plt.close(fig)

# Broadside
plot_empymod_vs_analytical(
    r_empymod, amp_bs, phase_bs, re_bs, im_bs,
    y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs,
    'Broadside', '17_4_9_empymod_vs_analytical_broadside.png'
)

# Endfire
plot_empymod_vs_analytical(
    r_empymod, amp_ef, phase_ef, re_ef, im_ef,
    x_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef,
    'Endfire', '18_4_9_empymod_vs_analytical_endfire.png'
)

# Oblique
plot_empymod_vs_analytical(
    r_empymod, amp_ob, phase_ob, re_ob, im_ob,
    x_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob,
    'Oblique', '19_4_9_empymod_vs_analytical_oblique.png'
)

def line_differences(r_other_line, r_analytical_line,
                             amp_other_line, phase_other_line, real_other_line, imag_other_line, 
                             amp_analytical_line, phase_analytical_line, real_analytical_line, imag_analytical_line):
    
    # Find the minimum bounds between the two lines
    min_r = max(min(r_other_line), min(r_analytical_line))
    max_r = min(max(r_other_line), max(r_analytical_line))

    # Find the line indices that are within the bounds
    idx_other = np.where((r_other_line >= min_r) & (r_other_line <= max_r))[0]
    idx_analytical = np.where((r_analytical_line >= min_r) & (r_analytical_line <= max_r))[0]
    print(f"Bounds: {min_r} to {max_r}")

    # Select the data within the bounds
    r_other_line = r_other_line[idx_other]
    r_analytical_line = r_analytical_line[idx_analytical]
    amp_other_line = amp_other_line[idx_other]
    amp_analytical_line = amp_analytical_line[idx_analytical]
    phase_other_line = phase_other_line[idx_other]
    phase_analytical_line = phase_analytical_line[idx_analytical]
    real_other_line = real_other_line[idx_other]
    real_analytical_line = real_analytical_line[idx_analytical]
    imag_other_line = imag_other_line[idx_other]
    imag_analytical_line = imag_analytical_line[idx_analytical]

    # Create a new array of r values that has 100 points between the bounds
    r = np.linspace(min_r, max_r, 100)

    # Interpolate both lines to the new r values
    amp_analytical_line = np.interp(r, r_analytical_line, amp_analytical_line)
    phase_analytical_line = np.interp(r, r_analytical_line, phase_analytical_line)
    real_analytical_line = np.interp(r, r_analytical_line, real_analytical_line)
    imag_analytical_line = np.interp(r, r_analytical_line, imag_analytical_line)

    amp_other_line = np.interp(r, r_other_line, amp_other_line)
    phase_other_line = np.interp(r, r_other_line, phase_other_line)
    real_other_line = np.interp(r, r_other_line, real_other_line)
    imag_other_line = np.interp(r, r_other_line, imag_other_line)

    diff_amp_percent = ((amp_other_line - amp_analytical_line) / amp_analytical_line) * 100
    diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100
    diff_phase_percent = (phase_other_line - phase_analytical_line) * 100
    diff_real_percent = ((real_other_line - real_analytical_line) / real_analytical_line) * 100
    diff_imag_percent = ((imag_other_line - imag_analytical_line) / imag_analytical_line) * 100

    analytic_data = np.column_stack((amp_analytical_line, phase_analytical_line, real_analytical_line, imag_analytical_line))
    other_data = np.column_stack((amp_other_line, phase_other_line, real_other_line, imag_other_line))
    diff_data = np.column_stack((diff_amp_percent, diff_amp_percent_power, diff_phase_percent, diff_real_percent, diff_imag_percent))

    return r, analytic_data, other_data, diff_data



def plot_differences_line(diff_amp, diff_phase, diff_real, diff_imag, r, type_rec, index, purpose):
    """
    Plot differences in amplitude, phase, real, and imaginary parts.
    """
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Amplitude difference
    axes[0, 0].plot(r, diff_amp, linestyle='-', color='blue', label='Amplitude Difference (%)', linewidth=2)
    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel('Amplitude Difference (%)', fontsize=12)
    axes[0, 0].set_title('Amplitude Difference', fontsize=18)
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    # Phase difference
    axes[0, 1].plot(r, diff_phase, linestyle='-', color='orange', label='Phase Difference (%)', linewidth=2)
    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel('Phase Difference (%)', fontsize=12)
    axes[0, 1].set_title('Phase Difference', fontsize=18)
    axes[0, 1].grid(True)

    # Real part difference
    if diff_real is not None:
        axes[1, 0].plot(r, diff_real, linestyle='-', color='green', label='Real Part Difference (%)', linewidth=2)
        axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
        axes[1, 0].set_ylabel('Real Part Difference (%)', fontsize=12)
        axes[1, 0].set_title('Real Part Difference', fontsize=18)
        axes[1, 0].grid(True)

    # Imaginary part difference
    if diff_imag is not None:
        axes[1, 1].plot(r, diff_imag, linestyle='-', color='red', label='Imaginary Part Difference (%)', linewidth=2)
        axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
        axes[1, 1].set_ylabel('Imaginary Part Difference (%)', fontsize=12)
        axes[1, 1].set_title('Imaginary Part Difference', fontsize=18)
    if diff_real is not None:
        axes[1, 1].grid(True)
    else:
        axes[1, 1].axis('off')
    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data Differences'.format(type_rec)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{type_rec}_differences_empymod.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Compute differences for each receiver type
r_diff_ef, analytic_ef, other_ef, diff_ef = line_differences(
    r_empymod, r[2:], amp_ef, phase_ef, re_ef, im_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef)
Evert_ef_amp, Evert_ef_phase, Evert_ef_re, Evert_ef_im = analytic_ef[:, 0], analytic_ef[:, 1], analytic_ef[:, 2], analytic_ef[:, 3]
EM_ef_amp, EM_ef_phase, EM_ef_re, EM_ef_im = other_ef[:, 0], other_ef[:, 1], other_ef[:, 2], other_ef[:, 3]
Diff_ef_amp, Diff_ef_amp10, Diff_ef_phase, Diff_ef_re, Diff_ef_im = diff_ef[:, 0], diff_ef[:, 1], diff_ef[:, 2], diff_ef[:, 3], diff_ef[:, 4]

r_diff_bs, analytic_bs, other_bs, diff_bs = line_differences(
    r_empymod, r[2:], amp_bs, phase_bs, re_bs, im_bs, abs_Ex_an_bs, phase_Ex_an_bs, real_Ex_an_bs, imag_Ex_an_bs)
Evert_bs_amp, Evert_bs_phase, Evert_bs_re, Evert_bs_im = analytic_bs[:, 0], analytic_bs[:, 1], analytic_bs[:, 2], analytic_bs[:, 3]
EM_bs_amp, EM_bs_phase, EM_bs_re, EM_bs_im = other_bs[:, 0], other_bs[:, 1], other_bs[:, 2], other_bs[:, 3]
Diff_bs_amp, Diff_bs_amp10, Diff_bs_phase, Diff_bs_re, Diff_bs_im = diff_bs[:, 0], diff_bs[:, 1], diff_bs[:, 2], diff_bs[:, 3], diff_bs[:, 4]

r_diff_ob, analytic_ob, other_ob, diff_ob = line_differences(
    r_empymod, r[2:], amp_ob, phase_ob, re_ob, im_ob, abs_Ex_an_ob, phase_Ex_an_ob, real_Ex_an_ob, imag_Ex_an_ob)
Evert_ob_amp, Evert_ob_phase, Evert_ob_re, Evert_ob_im = analytic_ob[:, 0], analytic_ob[:, 1], analytic_ob[:, 2], analytic_ob[:, 3]
EM_ob_amp, EM_ob_phase, EM_ob_re, EM_ob_im = other_ob[:, 0], other_ob[:, 1], other_ob[:, 2], other_ob[:, 3]
Diff_ob_amp, Diff_ob_amp10, Diff_ob_phase, Diff_ob_re, Diff_ob_im = diff_ob[:, 0], diff_ob[:, 1], diff_ob[:, 2], diff_ob[:, 3], diff_ob[:, 4]

# Plot differences for each receiver type
plot_differences_line(
    Diff_ef_amp, Diff_ef_phase, Diff_ef_re, Diff_ef_im, r_diff_ef, 'Endfire', 20, '4_9_empymod_vs_analytical')
plot_differences_line(
    Diff_bs_amp, Diff_bs_phase, Diff_bs_re, Diff_bs_im, r_diff_bs, 'Broadside', 21, '4_9_empymod_vs_analytical')
plot_differences_line(
    Diff_ob_amp, Diff_ob_phase, Diff_ob_re, Diff_ob_im, r_diff_ob, 'Oblique', 22, '4_9_empymod_vs_analytical')

Bounds: 0.2 to 4.75
Bounds: 0.2 to 4.75
Bounds: 0.2 to 4.75


C:\Users\Knight\AppData\Local\Temp\ipykernel_13772\3256241153.py:180: RuntimeWarning: overflow encountered in power
  diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100
C:\Users\Knight\AppData\Local\Temp\ipykernel_13772\3256241153.py:180: RuntimeWarning: invalid value encountered in subtract
  diff_amp_percent_power = ((np.power(10, amp_other_line) - np.power(10, amp_analytical_line)) / np.power(10, amp_analytical_line)) * 100


## Wideband Simulation

### Air

In [103]:
# total number of frequencies in input file
num_freq = 11
# number of receivers in input file - of each type
num_rec = 15

# Data
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
air_band_file = os.path.join(base_folder, "out_air_band_fc_100MHz", "electric_fields.txt")
air_band_data = np.loadtxt(air_band_file)
rec_file = os.path.join(base_folder, "out_air_band_fc_100MHz", "electric_fields_receiver_line.txt")
rec_data = np.loadtxt(rec_file)

# Receiver Radii
r_ef = rec_data[:num_rec, 0]
r_bs = rec_data[num_rec:2*num_rec, 1]
r_ob = np.sqrt(rec_data[2*num_rec:3*num_rec, 0]**2 + rec_data[2*num_rec:3*num_rec, 1]**2)
r_ob = np.round(r_ob, 2)

# Field Components
r_Ex = air_band_data[:, 0]
f_list = air_band_data[:, 3]
re_Ex = air_band_data[:, 4]
im_Ex = air_band_data[:, 5]
abs_Ex = np.abs(re_Ex + 1j * im_Ex)
phase_Ex = np.angle(re_Ex + 1j * im_Ex)  # returns phase in radians (−π to π)

# Extracting the data for an arbitrary receiver number
nr = 7
r_10 = r_Ex[(nr-1)*num_freq:(nr)*num_freq]
f_10 = f_list[(nr-1)*num_freq:(nr)*num_freq]
re_Ex_rec_10 = re_Ex[(nr-1)*num_freq:(nr)*num_freq]
im_Ex_rec_10 = im_Ex[(nr-1)*num_freq:(nr)*num_freq]
abs_Ex_rec_10 = abs_Ex[(nr-1)*num_freq:(nr)*num_freq]
phase_Ex_rec_10 = phase_Ex[(nr-1)*num_freq:(nr)*num_freq]
print(r_10)

[2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2]


In [104]:
def plot_wideband_receiver(frequencies, real_part, imag_part, abs_val, phase_val, receiver_num, save_path):
    """
    Plot wideband electric field data for a single receiver, marking all data points.
    """
    fig, axs = plt.subplots(2, 2, figsize=(10, 10), sharex=True)

    # Convert frequencies to MHz for x-axis
    freqs_MHz = frequencies / 1e6

    # Real Part
    axs[0, 0].plot(freqs_MHz, real_part, color='b', label='Real')
    axs[0, 0].scatter(freqs_MHz, real_part, color='b', marker='o')
    axs[0, 0].set_ylabel('Electric Field (V/m)')
    axs[0, 0].set_title('Real Part')
    # axs[0, 0].set_xscale('log')
    axs[0, 0].grid()
    axs[0, 0].set_xticks([freqs_MHz[0], freqs_MHz[-1]])
    axs[0, 0].set_xticklabels([f"{freqs_MHz[0]:.1f}", f"{freqs_MHz[-1]:.1f}"])

    # Imaginary Part
    axs[0, 1].plot(freqs_MHz, imag_part, color='r', label='Imag')
    axs[0, 1].scatter(freqs_MHz, imag_part, color='r', marker='o')
    axs[0, 1].set_ylabel('Electric Field (V/m)')
    axs[0, 1].set_title('Imaginary Part')
    # axs[0, 1].set_xscale('log')
    axs[0, 1].grid()
    axs[0, 1].set_xticks([freqs_MHz[0], freqs_MHz[-1]])
    axs[0, 1].set_xticklabels([f"{freqs_MHz[0]:.1f}", f"{freqs_MHz[-1]:.1f}"])

    # Absolute Value
    axs[1, 0].plot(freqs_MHz, abs_val, color='g', label='Abs')
    axs[1, 0].scatter(freqs_MHz, abs_val, color='g', marker='o')
    axs[1, 0].set_ylabel('Electric Field (V/m)')
    axs[1, 0].set_title('Absolute Value')
    # axs[1, 0].set_xscale('log')
    axs[1, 0].set_yscale('log')
    axs[1, 0].grid()
    axs[1, 0].set_xticks([freqs_MHz[0], freqs_MHz[-1]])
    axs[1, 0].set_xticklabels([f"{freqs_MHz[0]:.1f}", f"{freqs_MHz[-1]:.1f}"])

    # Phase
    axs[1, 1].plot(freqs_MHz, phase_val, color='m', label='Phase')
    axs[1, 1].scatter(freqs_MHz, phase_val, color='m', marker='o')
    axs[1, 1].set_ylabel('Phase (radians)')
    axs[1, 1].set_title('Phase')
    # axs[1, 1].set_xscale('log')
    axs[1, 1].grid()
    axs[1, 1].set_xticks([freqs_MHz[0], freqs_MHz[-1]])
    axs[1, 1].set_xticklabels([f"{freqs_MHz[0]:.1f}", f"{freqs_MHz[-1]:.1f}"])

    for ax in axs[1, :]:
        ax.set_xlabel('Frequency (MHz)')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.suptitle(f'Ex Field Component at Receiver {receiver_num}', fontsize=16, fontweight='bold')
    plt.savefig(save_path, dpi=300)
    plt.close()

In [105]:
plot_wideband_receiver(
    f_10, re_Ex_rec_10, im_Ex_rec_10, abs_Ex_rec_10, phase_Ex_rec_10, nr,
    os.path.join(postprocess_folder, f'23_wideband_air_r_{nr}.png')
)

### Half-Space

In [106]:
# total number of frequencies in input file
num_freq = 11
# number of receivers in input file - of each type
num_rec = 15

# Data
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
air_band_file = os.path.join(base_folder, "out_4_band_fc_100MHz", "electric_fields.txt")
air_band_data = np.loadtxt(air_band_file)
rec_file = os.path.join(base_folder, "out_4_band_fc_100MHz", "electric_fields_receiver_line.txt")
rec_data = np.loadtxt(rec_file)

# Receiver Radii
r_ef = rec_data[:num_rec, 0]
r_bs = rec_data[num_rec:2*num_rec, 1]
r_ob = np.sqrt(rec_data[2*num_rec:3*num_rec, 0]**2 + rec_data[2*num_rec:3*num_rec, 1]**2)
r_ob = np.round(r_ob, 2)

# Field Components
r_Ex = air_band_data[:, 0]
f_list = air_band_data[:, 3]
re_Ex = air_band_data[:, 4]
im_Ex = air_band_data[:, 5]
abs_Ex = np.abs(re_Ex + 1j * im_Ex)
phase_Ex = np.angle(re_Ex + 1j * im_Ex)  # returns phase in radians (−π to π)

# Extracting the data for an arbitrary receiver number
nr = 7
r_10 = r_Ex[(nr-1)*num_freq:(nr)*num_freq]
f_10 = f_list[(nr-1)*num_freq:(nr)*num_freq]
re_Ex_rec_10 = re_Ex[(nr-1)*num_freq:(nr)*num_freq]
im_Ex_rec_10 = im_Ex[(nr-1)*num_freq:(nr)*num_freq]
abs_Ex_rec_10 = abs_Ex[(nr-1)*num_freq:(nr)*num_freq]
phase_Ex_rec_10 = phase_Ex[(nr-1)*num_freq:(nr)*num_freq]
print(r_10)

[2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2]


In [107]:
plot_wideband_receiver(
    f_10, re_Ex_rec_10, im_Ex_rec_10, abs_Ex_rec_10, phase_Ex_rec_10, nr,
    os.path.join(postprocess_folder, f'24_wideband_4_r_{nr}.png')
)

### Two Layers

In [108]:
# total number of frequencies in input file
num_freq = 11
# number of receivers in input file - of each type
num_rec = 15

# Data
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
air_band_file = os.path.join(base_folder, "out_4_9_band_fc_100MHz", "electric_fields.txt")
air_band_data = np.loadtxt(air_band_file)
rec_file = os.path.join(base_folder, "out_4_9_band_fc_100MHz", "electric_fields_receiver_line.txt")
rec_data = np.loadtxt(rec_file)

# Receiver Radii
r_ef = rec_data[:num_rec, 0]
r_bs = rec_data[num_rec:2*num_rec, 1]
r_ob = np.sqrt(rec_data[2*num_rec:3*num_rec, 0]**2 + rec_data[2*num_rec:3*num_rec, 1]**2)
r_ob = np.round(r_ob, 2)

# Field Components
r_Ex = air_band_data[:, 0]
f_list = air_band_data[:, 3]
re_Ex = air_band_data[:, 4]
im_Ex = air_band_data[:, 5]
abs_Ex = np.abs(re_Ex + 1j * im_Ex)
phase_Ex = np.angle(re_Ex + 1j * im_Ex)  # returns phase in radians (−π to π)

# Extracting the data for an arbitrary receiver number
nr = 7
r_10 = r_Ex[(nr-1)*num_freq:(nr)*num_freq]
f_10 = f_list[(nr-1)*num_freq:(nr)*num_freq]
re_Ex_rec_10 = re_Ex[(nr-1)*num_freq:(nr)*num_freq]
im_Ex_rec_10 = im_Ex[(nr-1)*num_freq:(nr)*num_freq]
abs_Ex_rec_10 = abs_Ex[(nr-1)*num_freq:(nr)*num_freq]
phase_Ex_rec_10 = phase_Ex[(nr-1)*num_freq:(nr)*num_freq]
print(r_10)

plot_wideband_receiver(
    f_10, re_Ex_rec_10, im_Ex_rec_10, abs_Ex_rec_10, phase_Ex_rec_10, nr,
    os.path.join(postprocess_folder, f'25_wideband_4_9_r_{nr}.png')
)

[2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2]
